# Sprint 2 — Ingeniería de datos y pipeline reproducible · Olist

**Proyecto:** Análisis de Propensión al Churn de Clientes — Olist Brazilian E-Commerce
**Sprint 2:** pipeline de datos reproducible sobre **Azure SQL Database** · Horizonte **H = 90 días**

---

## Objetivo
Construir un **pipeline reproducible** que, parametrizado por una **fecha de corte `t`**, genere
la tabla maestra a nivel de cliente (features + target) leyendo desde **Azure SQL**, y simule la
**llegada mensual de datos** filtrando por fecha en la propia base. Sigue el plan maestro
(secciones 7 y 10) y los hallazgos del Sprint 1.

## Decisiones heredadas del Sprint 1
- **Grano:** `customer_unique_id`. **Sin fuga:** features con datos ≤ t, target con datos > t.
- **Target accionable (`compra_futura`):** 1 si el cliente compra en `(t, t+H]` (propensión a
  recompra en la ventana). Es el complemento del `is_churn` por ventana, que se conserva como
  métrica de negocio (degenerado ~97-99 %, por eso se modela la clase positiva con `class_weight`).
- **Features:** RFM (recencia, frecuencia, monetario), tenure, nº de ítems, categoría, tipo de
  pago, reseña media, retraso medio, % entregas tardías y distancia vendedor–cliente.
- **Partición:** estrictamente temporal (out-of-time): train / validación / backtest por snapshot.

## 1. Configuración

In [ ]:
import warnings, os, getpass, urllib.parse, subprocess, sys
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')

H            = 90
RANDOM_STATE = 42
ESTADOS_INVALIDOS = ['canceled','unavailable']
# Features de modelado
NUM = ['recencia','frecuencia','monetario','tenure','n_items_medio','review_mean',
       'retraso_medio','pct_tardias','dist_media']
CAT = ['estado','categoria','tipo_pago']
# Cortes mensuales para la simulación (la ventana t+H debe caer dentro del rango de datos)
CORTES = pd.date_range('2017-06-30','2018-07-31', freq='ME')
print('H =', H, '| snapshots:', len(CORTES))

## 2. Conexión a Azure SQL Database

Lee servidor/base/usuario de `sql_conexion.txt` (generado por `azure_sql_setup.ps1`) y pide la
contraseña con `getpass`. Requiere `sqlalchemy`, `pyodbc` y el ODBC Driver 18.

In [ ]:
try:
    from sqlalchemy import create_engine, text
    import pyodbc
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','sqlalchemy','pyodbc','-q'])
    from sqlalchemy import create_engine, text
    import pyodbc

SQL_SERVER='olist-sql-40548.database.windows.net'; SQL_DB='OlistDB'; SQL_ADMIN='olistadmin'
if os.path.exists('sql_conexion.txt'):
    for _l in open('sql_conexion.txt',encoding='utf-8'):
        if '=' in _l:
            _k,_v=_l.strip().split('=',1)
            if _k=='SQL_SERVER': SQL_SERVER=_v
            if _k=='SQL_DB': SQL_DB=_v
            if _k=='SQL_ADMIN': SQL_ADMIN=_v
PWD=os.environ.get('AZURE_SQL_PASSWORD') or getpass.getpass(f'Contraseña de {SQL_ADMIN}: ')
driver='ODBC Driver 18 for SQL Server'
if driver not in pyodbc.drivers(): driver='ODBC Driver 17 for SQL Server'
odbc=(f'Driver={{{driver}}};Server=tcp:{SQL_SERVER},1433;Database={SQL_DB};'
      f'Uid={SQL_ADMIN};Pwd={PWD};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60;')
engine=create_engine('mssql+pyodbc:///?odbc_connect='+urllib.parse.quote_plus(odbc))

def leer_tabla(nombre, where=None, cols='*'):
    """Lee una tabla de Azure SQL; `where` permite filtrar EN LA FUENTE (simulación mensual)."""
    q=f'SELECT {cols} FROM {nombre}'+(f' WHERE {where}' if where else '')
    return pd.read_sql(q, engine)

with engine.connect() as _cx:
    print('Conectado a', SQL_SERVER,'/',SQL_DB)
    print('Tablas:', [r[0] for r in _cx.execute(text('SELECT name FROM sys.tables ORDER BY name'))])

## 3. Lectura parametrizada por fecha (simulación de la llegada mensual)

El plan maestro pide simular la llegada de datos mensuales **filtrando por fecha en la fuente**.
La función `leer_tabla` lo permite con una cláusula `WHERE` que se ejecuta en SQL, no en pandas.
Ejemplo: pedidos hasta una fecha de corte.

In [ ]:
demo = leer_tabla('orders', where="order_purchase_timestamp <= '2018-05-31'",
                  cols='order_id, customer_id, order_purchase_timestamp, order_status')
print('Pedidos hasta 2018-05-31 (filtrado en SQL):', len(demo))
demo.head(3)

## 4. Limpieza y tabla pedido enriquecida

Lectura de las tablas desde SQL, limpieza (estados válidos, fechas, faltantes) y construcción
de una tabla **a nivel de pedido** con valor, nº de ítems, reseña, métricas de entrega,
categoría, tipo de pago y distancia vendedor–cliente. Se construye una vez y se reutiliza.

In [ ]:
# Carga de tablas (una vez)
orders   = leer_tabla('orders');         customers=leer_tabla('customers')
items    = leer_tabla('order_items');    payments =leer_tabla('order_payments')
reviews  = leer_tabla('order_reviews');  products =leer_tabla('products')
sellers  = leer_tabla('sellers');        cat_trans=leer_tabla('category_translation')
geo      = leer_tabla('geolocation').rename(columns={'geolocation_zip_code_prefix':'zip_prefix'})
for c in ['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']:
    orders[c]=pd.to_datetime(orders[c], errors='coerce')

def dominante(df,key,col):
    c=(df.dropna(subset=[col]).groupby([key,col]).size().reset_index(name='n')
         .sort_values('n').drop_duplicates(key,keep='last'))
    return c.set_index(key)[col]
def haversine(a,b,c,d):
    R=6371.0;p1,p2=np.radians(a),np.radians(c);dphi=np.radians(c-a);dl=np.radians(d-b)
    x=np.sin(dphi/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2;return 2*R*np.arcsin(np.sqrt(x))

def pedidos_enriquecidos():
    oc=orders.merge(customers,on='customer_id',how='left'); oc=oc[~oc.order_status.isin(ESTADOS_INVALIDOS)].copy()
    val=items.groupby('order_id').agg(price=('price','sum'),freight=('freight_value','sum'),n_items=('order_item_id','max')).reset_index(); val['valor']=val.price+val.freight
    pay=payments.sort_values('payment_value',ascending=False).groupby('order_id').agg(tipo_pago=('payment_type','first')).reset_index()
    rev=reviews.groupby('order_id')['review_score'].mean().reset_index()
    itc=items.merge(products[['product_id','product_category_name']],on='product_id',how='left').merge(cat_trans,on='product_category_name',how='left')
    itc['categoria']=itc['product_category_name_english'].fillna(itc['product_category_name']); cat=dominante(itc,'order_id','categoria').reset_index()
    gs=geo.rename(columns={'zip_prefix':'seller_zip_code_prefix','lat':'s_lat','lng':'s_lng'}); gc=geo.rename(columns={'zip_prefix':'customer_zip_code_prefix','lat':'c_lat','lng':'c_lng'})
    its=items.merge(sellers[['seller_id','seller_zip_code_prefix']],on='seller_id',how='left').merge(gs[['seller_zip_code_prefix','s_lat','s_lng']],on='seller_zip_code_prefix',how='left')
    its=its.merge(oc[['order_id','customer_zip_code_prefix']],on='order_id',how='left').merge(gc[['customer_zip_code_prefix','c_lat','c_lng']],on='customer_zip_code_prefix',how='left')
    its['dist_km']=haversine(its.s_lat,its.s_lng,its.c_lat,its.c_lng); dist=its.groupby('order_id')['dist_km'].mean().reset_index()
    oe=oc[['order_id','customer_unique_id','customer_state','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']].copy()
    oe['retraso_dias']=(oe.order_delivered_customer_date-oe.order_estimated_delivery_date).dt.days
    oe['tardio']=(oe.retraso_dias>0).astype('float'); oe.loc[oe.order_delivered_customer_date.isna(),'tardio']=np.nan
    return (oe.merge(val[['order_id','valor','n_items']],on='order_id',how='left').merge(rev,on='order_id',how='left')
              .merge(cat,on='order_id',how='left').merge(pay,on='order_id',how='left').merge(dist,on='order_id',how='left'))

OE=pedidos_enriquecidos(); print('Tabla pedido enriquecida:', OE.shape); OE.head(3)

## 5. Feature engineering parametrizado por la fecha de corte

`construir_tabla_maestra(t, H)` calcula, a nivel de cliente, las features con datos **≤ t** y el
target `compra_futura` con datos en **`(t, t+H]`**. Es el corazón reproducible del pipeline:
una misma función genera el dataset de cualquier mes (sin fuga de información).

### Diccionario de features

Todas se calculan **por cliente** (`customer_unique_id`) usando solo datos **hasta la fecha de corte `t`** (sin fuga).

**Núcleo RFM**

| Feature | Tipo | Descripción |
|---|---|---|
| `recencia` | num | Días desde la última compra del cliente hasta `t` (más alto = lleva más sin comprar). |
| `frecuencia` | num | Número de pedidos del cliente hasta `t`. |
| `monetario` | num | Gasto acumulado hasta `t` (suma de `price + freight_value`). |

**Derivadas de comportamiento**

| Feature | Tipo | Descripción |
|---|---|---|
| `tenure` | num | Antigüedad: días desde la **primera** compra hasta `t`. |
| `n_items_medio` | num | Promedio de ítems por pedido (tamaño de canasta). |
| `review_mean` | num | Puntuación media de reseñas (1–5). *Variable de control.* |
| `retraso_medio` | num | Días medios de retraso en entrega (entregado − estimado; + = tarde). *Control.* |
| `pct_tardias` | num | Proporción de pedidos entregados tarde (0–1). *Control.* |
| `dist_media` | num | Distancia media vendedor–cliente en km (haversine). |

**Categóricas**

| Feature | Tipo | Descripción |
|---|---|---|
| `estado` | cat | Estado (UF) del cliente — patrón geográfico. |
| `categoria` | cat | Categoría de producto dominante. |
| `tipo_pago` | cat | Tipo de pago dominante (credit_card, boleto, voucher, debit_card). |

**Objetivo / métrica**

| Columna | Descripción |
|---|---|
| `compra_futura` | **Target.** 1 si el cliente compra en `(t, t+90]` (propensión a recompra). |
| `is_churn` | Métrica de negocio = `1 − compra_futura`. |

> Preprocesamiento en el pipeline: numéricas → imputación por mediana + `StandardScaler`; categóricas → imputación por moda + `OneHotEncoder`.

In [ ]:
def construir_tabla_maestra(t, H=90):
    t=pd.Timestamp(t); obs=OE[OE.order_purchase_timestamp<=t]
    fut=OE[(OE.order_purchase_timestamp>t)&(OE.order_purchase_timestamp<=t+pd.Timedelta(days=H))]
    g=obs.groupby('customer_unique_id')
    df=pd.DataFrame({'recencia':(t-g.order_purchase_timestamp.max()).dt.days,'frecuencia':g.order_id.nunique(),
        'monetario':g.valor.sum(),'tenure':(t-g.order_purchase_timestamp.min()).dt.days,'n_items_medio':g.n_items.mean(),
        'review_mean':g.review_score.mean(),'retraso_medio':g.retraso_dias.mean(),'pct_tardias':g.tardio.mean(),'dist_media':g.dist_km.mean()})
    df['estado']=dominante(obs,'customer_unique_id','customer_state')
    df['categoria']=dominante(obs,'customer_unique_id','categoria')
    df['tipo_pago']=dominante(obs,'customer_unique_id','tipo_pago')
    df['compra_futura']=df.index.isin(set(fut.customer_unique_id)).astype(int)   # target accionable
    df['is_churn']=1-df['compra_futura']                                          # métrica de negocio
    df['snapshot']=t
    return df

ej=construir_tabla_maestra('2018-05-31', H)
print('Tabla maestra (corte 2018-05-31):', ej.shape, '| tasa compra_futura:', round(ej.compra_futura.mean(),4))
ej.head(3)

## 6. Pipeline de modelado reproducible (sklearn)

Separación de responsabilidades: la construcción de la tabla maestra (arriba) y las
**transformaciones de modelado** (imputación, escalado y encoding) dentro de un
`sklearn.Pipeline`, evitando fuga entre train y test. El estimador final se afina en el Sprint 3;
aquí se usa una regresión logística con `class_weight='balanced'` como baseline.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

pre=ColumnTransformer([
    ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),NUM),
    ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),CAT)])
pipe=Pipeline([('pre',pre),
               ('clf',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=RANDOM_STATE))])
pipe

## 7. Simulación de la llegada mensual y partición temporal

Para cada corte mensual se genera la tabla maestra; los snapshots se reparten en train /
validación / backtest según su fecha (out-of-time, sin solape).

In [ ]:
datasets={c:construir_tabla_maestra(c,H) for c in CORTES}
TR=pd.concat([datasets[c] for c in CORTES if c<=pd.Timestamp('2018-02-28')])
VA=pd.concat([datasets[c] for c in CORTES if pd.Timestamp('2018-02-28')<c<=pd.Timestamp('2018-04-30')])
BT=pd.concat([datasets[c] for c in CORTES if c>pd.Timestamp('2018-04-30')])
print(pd.DataFrame({'particion':['train','validación','backtest'],
                    'filas':[len(TR),len(VA),len(BT)],
                    'tasa_compra_futura':[round(TR.compra_futura.mean(),4),round(VA.compra_futura.mean(),4),round(BT.compra_futura.mean(),4)]}))

## 8. Entrenamiento out-of-time y métricas

Se entrena en train y se evalúa en validación y backtest. Por el fuerte desbalance se reportan
**AUC, recall y average precision** (no accuracy), más métricas de negocio.

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, average_precision_score
pipe.fit(TR[NUM+CAT], TR['compra_futura'])
filas=[]
for nm,S in [('validación',VA),('backtest',BT)]:
    p=pipe.predict_proba(S[NUM+CAT])[:,1]; yhat=(p>=0.5).astype(int)
    filas.append((nm, round(roc_auc_score(S.compra_futura,p),3), round(recall_score(S.compra_futura,yhat),3),
                  round(average_precision_score(S.compra_futura,p),3)))
met=pd.DataFrame(filas,columns=['partición','AUC','recall','avg_precision']); print(met)

# Importancia (coeficientes) sobre las numéricas
import numpy as np
coef=pipe.named_steps['clf'].coef_[0][:len(NUM)]
print('\nDirección de las features numéricas (coef LogReg):')
for f,c in sorted(zip(NUM,coef), key=lambda z:-abs(z[1])): print(f'  {f:14s} {c:+.3f}')

# Métricas de negocio (backtest)
print('\n--- Negocio (backtest) ---')
print('Clientes evaluados:', len(BT), '| tasa de recompra (compra_futura):', round(BT.compra_futura.mean(),4))
print('Valor monetario histórico en clientes que recompran: R$ {:,.0f}'.format(BT.loc[BT.compra_futura==1,'monetario'].sum()))

## 9. Persistencia y versionado del pipeline

El pipeline entrenado se serializa con `joblib` para reutilizarlo en Sprint 3 (selección de
modelo) y en producción (retraining mensual). El código se versiona con Git.

In [ ]:
import joblib
joblib.dump(pipe, 'pipeline_churn_sprint2.pkl')
print('Pipeline guardado en pipeline_churn_sprint2.pkl')
# Para reutilizarlo:  pipe = joblib.load('pipeline_churn_sprint2.pkl')

## 10. Conclusiones y handoff a Sprint 3

- Pipeline **reproducible y parametrizado por fecha** que lee de Azure SQL, construye la tabla
  maestra a nivel de cliente y simula la llegada mensual filtrando por fecha en la fuente.
- Target accionable `compra_futura` (recompra en la ventana); `is_churn` se reporta como métrica
  de negocio. El desbalance se maneja con `class_weight` y métricas AUC/recall/AP.
- La **recencia** y la **frecuencia** son los predictores dominantes; la experiencia (entrega,
  reseña) aporta poco, en línea con el Sprint 1.
- **Sprint 3:** selección y tuning del estimador (árboles / gradient boosting), calibración de
  probabilidades, umbral por valor de negocio e integración con MLflow.

## 11. Verificación y reproducibilidad

In [ ]:
# 1) Sin fuga: features de un snapshot solo usan datos <= t
t0=pd.Timestamp('2018-05-31'); obs=OE[OE.order_purchase_timestamp<=t0]
assert obs.order_purchase_timestamp.max()<=t0, 'Fuga temporal'
# 2) Partición temporal sin solape
assert TR.snapshot.max()<VA.snapshot.min()<=VA.snapshot.max()<BT.snapshot.min(), 'Solape de snapshots'
# 3) Target binario y grano correcto
assert set(ej.compra_futura.unique())<={0,1} and ej.index.is_unique, 'Target/grano inválido'
# 4) El pipeline produce probabilidades válidas
import numpy as np
p=pipe.predict_proba(VA[NUM+CAT])[:,1]; assert ((p>=0)&(p<=1)).all(), 'Probabilidades fuera de [0,1]'
print('OK - Pipeline Sprint 2 reproducible y sin fuga temporal.')